<a href="https://colab.research.google.com/github/kondratyukkatya9-ML/ML_home_work/blob/main/Copy_of_HW_2_4_kNN_%D0%9A%D1%80%D0%BE%D1%81%D0%B2%D0%B0%D0%BB%D1%96%D0%B4%D0%B0%D1%86%D1%96%D1%8F_%D1%96_%D1%82%D1%8E%D0%BD%D0%B8%D0%BD%D0%B3_%D0%B3%D1%96%D0%BF%D0%B5%D1%80%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням kNearestNeighboors, знайдемо оптимальні гіперпараметри для цього методу і зробимо базові ансамблі. Це дасть змогу порівняти перформанс моделі з попередніми вивченими методами.

0. Зчитайте дані `train.csv` та зробіть препроцесинг використовуючи написаний Вами скрипт `process_bank_churn.py` так, аби в результаті отримати дані в розбитті X_train, train_targets, X_val, val_targets для експериментів.

  Якщо Вам не вдалось реалізувати в завданні `2.3. Дерева прийняття рішень` скрипт `process_bank_churn.py` - можна скористатись готовим скриптом з запропонованого рішення того завдання.

In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!wget -q -O process_bank_churn.py https://raw.githubusercontent.com/kondratyukkatya9-ML/ML_home_work/main/process_bank_churn.py

In [4]:
raw_df = pd.read_csv('/content/drive/MyDrive/kaggle_hw/train.csv', index_col=0)

In [5]:
import process_bank_churn as pbc

In [6]:
data = pbc.preprocess_data(raw_df, scaler_numeric=True)

In [7]:
X_train, train_targets = data['train_X'], data['train_y']
X_val, val_targets = data['val_X'], data['val_y']

In [8]:
X_train.shape, X_val.shape
X_train.describe().T[['mean', 'std']]

,mean,std
CreditScore,0.543771,0.172637
Age,0.351570,0.145329
Tenure,0.502608,0.278197
Balance,0.205028,0.285586
NumOfProducts,0.196750,0.177549
HasCrCard,0.790333,0.407088
IsActiveMember,0.491583,0.499950
EstimatedSalary,0.589960,0.227969
Geography_France,0.600917,0.489730
Geography_Germany,0.179250,0.383577


1. Навчіть на цих даних класифікатор kNN з параметрами за замовченням і виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах. Зробіть заключення про отриману модель: вона хороша/погана, чи є high bias/high variance?

In [43]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

model_knn = KNeighborsClassifier()
model_knn.fit(X_train, train_targets)

train_auc_knn = roc_auc_score(train_targets, model_knn.predict_proba(X_train)[:, 1])
val_auc_knn = roc_auc_score(val_targets, model_knn.predict_proba(X_val)[:, 1])

In [44]:
print(f'Train AUROC: {train_auc_knn:.4f}')
print(f'Val AUROC:   {val_auc_knn:.4f}')

Train AUROC: 0.9559
Val AUROC:   0.8526


Модель дещо перенавчена, є  high variance - розрив між метриками на тренувальному наборі та тестовому.

2. Використовуючи `GridSearchCV` знайдіть оптимальне значення параметра `n_neighbors` для класифікатора `kNN`. Псотавте крос валідацію на 5 фолдів.

  Після успішного завершення пошуку оптимального гіперпараметра
    - виведіть найкраще значення параметра
    - збережіть в окрему змінну `knn_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `knn_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пукнтом (2) цього завдання? Чи є вона краще за дерево прийняття рішень з попереднього ДЗ?

In [11]:
from sklearn.model_selection import GridSearchCV

In [12]:
search_grid = GridSearchCV(
    model_knn,
    param_grid={'n_neighbors': range(1,51,2)},
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
)

In [13]:
search_grid.fit(X_train, train_targets)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'n_neighbors': range(1, 51, 2)}, scoring='roc_auc')

In [14]:
search_grid.best_params_

{'n_neighbors': 19}

In [15]:
knn_best  = search_grid.best_estimator_

In [16]:
train_auc_best = roc_auc_score(train_targets, knn_best.predict_proba(X_train)[:, 1])
val_auc_best = roc_auc_score(val_targets, knn_best.predict_proba(X_val)[:, 1])

In [17]:
print(f'Train AUROC: {train_auc_best:.4f}')
print(f'Val AUROC:   {val_auc_best:.4f}')

Train AUROC: 0.9227
Val AUROC:   0.8908


Розрив між метриками на тренувальному і валідаційному датасетах скоротився, модель стала караща, але метрики дерева вона не випередила.

3. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `GridSearchCV` за сіткою параметрів
  - `max_depth` від 1 до 20 з кроком 2
  - `max_leaf_nodes` від 2 до 10 з кроком 1

  Обовʼязково при цьому ініціюйте модель з фіксацією `random_state`.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `dt_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли вручну?

In [18]:
from sklearn.tree import DecisionTreeClassifier

In [19]:
model_dt = DecisionTreeClassifier(random_state=42)

In [20]:
param_grid_dt = {
    'max_depth': range(1, 21, 2),
    'max_leaf_nodes': range(2, 11, 1),
}

In [21]:
search_grid_dt = GridSearchCV(
    model_dt,
    param_grid_dt,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
)

In [22]:
search_grid_dt.fit(X_train, train_targets)

GridSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(1, 21, 2),
                         'max_leaf_nodes': range(2, 11)},
             scoring='roc_auc')

In [23]:
search_grid_dt.best_params_

{'max_depth': 5, 'max_leaf_nodes': 10}

In [24]:
dt_best = search_grid_dt.best_estimator_

In [25]:
train_auc_best = roc_auc_score(train_targets, dt_best.predict_proba(X_train)[:, 1])
val_auc_best = roc_auc_score(val_targets, dt_best.predict_proba(X_val)[:, 1])

In [26]:
print(f'Train AUROC: {train_auc_best:.4f}')
print(f'Val AUROC:   {val_auc_best:.4f}')

Train AUROC: 0.9015
Val AUROC:   0.9002


так, модель вийшла кращою, метрики на валідації вищі, чим в попередньої, вона не перенавчена - метрики знаходяться дуже близько, але видно, що ми задали надто малий діапазон для пошуків оптимальних параметрів

In [27]:
# Розширена сітка — власна гіпотеза, бо  значення max_leaf_nodes було на верхній межі заданого діапазону
param_grid_wide = {
    'max_depth': range(1, 21, 2),
    'max_leaf_nodes': range(2, 50, 1),
}

search_wide = GridSearchCV(model_dt, param_grid_wide, cv=3, scoring='roc_auc', n_jobs=-1)
search_wide.fit(X_train, train_targets)

GridSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(1, 21, 2),
                         'max_leaf_nodes': range(2, 50)},
             scoring='roc_auc')

In [28]:
search_wide.best_params_

{'max_depth': 5, 'max_leaf_nodes': 28}

In [29]:
wide_best = search_wide.best_estimator_

In [30]:
train_auc_best2 = roc_auc_score(train_targets, wide_best.predict_proba(X_train)[:, 1])
val_auc_best2 = roc_auc_score(val_targets, wide_best.predict_proba(X_val)[:, 1])

In [31]:
print(f'Train AUROC: {train_auc_best2:.4f}')
print(f'Val AUROC:   {val_auc_best2:.4f}')

Train AUROC: 0.9257
Val AUROC:   0.9220


як видно, більший діапазое дозволив знайти оптимальніші гіперпараметри і дозволив підняти якість моделі: Train AUROC: 0.9257, Val AUROC:   0.9220

4. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `RandomizedSearchCV` за заданою сіткою параметрів і кількість ітерацій 40.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, зафіксуйте `random_seed` процедури крос валідації та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_random_search_best` найкращу модель, знайдену з `RandomizedSearchCV`
    - оцініть якість передбачень  `dt_random_search_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли з `GridSearch`?
    - проаналізуйте параметри `dt_random_search_best` і порівняйте з параметрами `dt_best` - яку бачите відмінність? Ця вправа потрібна аби зрозуміти, як різні налаштування `DecisionTreeClassifier` впливають на якість моделі.

In [32]:
from sklearn.model_selection import RandomizedSearchCV

In [33]:
params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': np.arange(1, 20),
    'max_leaf_nodes': np.arange(2, 20),
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

In [34]:


random_search_dt = RandomizedSearchCV(
    model_dt,
    params_dt,
    n_iter=40,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
)

In [35]:
random_search_dt.fit(X_train, train_targets)

RandomizedSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=40, n_jobs=-1,
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19]),
                                        'max_features': [None, 'sqrt', 'log2'],
                                        'max_leaf_nodes': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
       19]),
                                        'min_samples_leaf': [1, 2, 4, 8],
                                        'min_samples_split': [2, 5, 10, 20],
                                        'splitter': ['best', 'random']},
                   random_state=42, scoring='roc_auc')

In [36]:
random_search_dt.best_params_

{'splitter': 'best',
 'min_samples_split': 20,
 'min_samples_leaf': 2,
 'max_leaf_nodes': np.int64(14),
 'max_features': None,
 'max_depth': np.int64(16),
 'criterion': 'entropy'}

In [37]:
random_search_best = random_search_dt.best_estimator_

In [38]:
train_auc_best3 = roc_auc_score(train_targets, random_search_best.predict_proba(X_train)[:, 1])
val_auc_best3 = roc_auc_score(val_targets, random_search_best.predict_proba(X_val)[:, 1])

In [39]:
print(f'Train AUROC: {train_auc_best3:.4f}')
print(f'Val AUROC:   {val_auc_best3:.4f}')

Train AUROC: 0.9169
Val AUROC:   0.9166


В попередній моделі ми оптимізували тільки 2 параметри: max_depth': 5, 'max_leaf_nodes': 28, в random_search_dt  ми оптимізували набагато більше параметрів, але можна порівняти max_leaf_nodes (було 28, стало 14) та max_depth (було 5, стало 16). Модель кращою не стала,GridSearch лише по двох параметрах, але в достатньому діапазоні, дав кращий скор, чим RandomSearch на багатьох параметрах. Random Search виконався набагато швидше ( 1 секунда, замість 5 при GridSearch)

5. Якщо у Вас вийшла метрика `AUROC` в цій серії експериментів - зробіть ще один `submission` на Kaggle і додайте код для цього і скріншот скора на публічному лідерборді нижче.

  Сподіваюсь на цьому етапі ви вже відчули себе справжнім дослідником 😉

In [40]:
test_df = pd.read_csv('/content/drive/MyDrive/kaggle_hw/test.csv')

In [41]:
X_test = pbc.preprocess_new_data(
    test_df,
    input_cols=data['input_cols'],
    encoder=data['encoder'],
    scaler=data['scaler'],
)

In [42]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'Exited': random_search_best.predict_proba(X_test)[:, 1],
})
submission.to_csv('submission_dt_random.csv', index=False)